In [2]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
from scipy.stats import gaussian_kde
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from libpysal.weights import lat2W
from libpysal.weights import Queen
from libpysal.weights import DistanceBand
from esda.moran import Moran_Local, Moran
from shapely.geometry import box

Create a dataframe with all spots that are active for at least six months with at at most 14 days between any two observations

In [4]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["date"] = pd.to_datetime(df["created_at_local"].dt.date)

# ignore standing water type and stream type categories because they do not require time series
df = df[~df["Category"].isin(["standing water type", "stream type"])]

# dates per spot and category
spot_dates = (
    df
    .groupby(["longitude", "latitude", "Category", "date"])
    .size()
    .reset_index(name="n_obs")
)

def get_all_streaks(dates):
    dates = sorted(set(dates))
    streaks = []
    start = dates[0]
    prev = dates[0]

    for d in dates[1:]:
        gap = (pd.Timestamp(d) - pd.Timestamp(prev)).days
        if gap <= 14:
            prev = d
        else:
            duration = (pd.Timestamp(prev) - pd.Timestamp(start)).days
            if duration >= 180: # about half a year
                streaks.append({
                    "streak_start": start,
                    "streak_end": prev,
                    "streak_days": duration
                })
            start = d
            prev = d

    # last streak
    duration = (pd.Timestamp(prev) - pd.Timestamp(start)).days
    if duration >= 180:
        streaks.append({
            "streak_start": start,
            "streak_end": prev,
            "streak_days": duration
        })

    return pd.DataFrame(streaks)

streak_periods = (
    spot_dates
    .groupby(["longitude", "latitude", "Category"])["date"]
    .apply(get_all_streaks)
    .reset_index(level=3, drop=True)
    .reset_index()
)

persistent = streak_periods.copy().reset_index(drop=True)
persistent["streak_start"] = pd.to_datetime(persistent["streak_start"])
persistent["streak_end"] = pd.to_datetime(persistent["streak_end"])

# country
spot_country = (
    df
    .groupby(["longitude", "latitude"])["Country"]
    .first()
    .reset_index()
)
persistent = persistent.merge(spot_country, on=["longitude", "latitude"], how="left")

# users active during the streak at this spot
df_merged = df.merge(
    persistent[["longitude", "latitude", "Category", "streak_start", "streak_end"]],
    on=["longitude", "latitude", "Category"],
    how="inner"
)

df_merged = df_merged[
    (df_merged["date"] >= df_merged["streak_start"]) &
    (df_merged["date"] <= df_merged["streak_end"])
]

spot_users = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])["created_by"]
    .nunique()
    .reset_index(name="n_users")
)
persistent = persistent.merge(spot_users, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

spot_user_ids = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])["created_by"]
    .apply(lambda x: [int(i) for i in x.unique()])
    .reset_index(name="user_ids")
)
persistent = persistent.merge(spot_user_ids, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

spot_obs = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])
    .size()
    .reset_index(name="n_obs")
)
persistent = persistent.merge(spot_obs, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

# final table
persistent = persistent[["latitude", "longitude", "Country", "streak_days", "streak_start", "streak_end", "n_users", "user_ids", "n_obs", "Category"]]
persistent = persistent.sort_values("streak_days", ascending=False).reset_index(drop=True)

print(persistent.head(20))
print(f"number of spots with >=180 days consecutive activity: {len(persistent)}")

persistent.to_csv("../Products/CSVs/persistent_spots_14d.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_52116\2540333354.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

     latitude   longitude      Country  streak_days streak_start streak_end  \
0   49.375012    8.888853      Germany       1822.0   2020-07-28 2025-07-24   
1   49.375062    8.889207      Germany       1822.0   2020-07-28 2025-07-24   
2   51.060439 -115.328485       Canada       1763.0   2019-08-02 2024-05-30   
3   47.726072   13.065014      Austria       1410.0   2017-12-25 2021-11-04   
4   47.394939    8.733537  Switzerland        791.0   2022-07-11 2024-09-09   
5   47.395523    8.730670  Switzerland        791.0   2022-07-11 2024-09-09   
6   47.394939    8.733537  Switzerland        761.0   2020-05-22 2022-06-22   
7   47.789612   13.068625      Austria        755.0   2018-08-17 2020-09-10   
8   48.328912   16.214674      Austria        711.0   2020-07-28 2022-07-09   
9   47.389403    8.561560  Switzerland        642.0   2020-11-03 2022-08-07   
10  47.387870    8.563581  Switzerland        642.0   2020-11-03 2022-08-07   
11  47.391386    8.560455  Switzerland        642.0 

And the same for spots that were active at least once a month for at least 12 months consecutively

In [5]:
df = pd.read_csv("../CWData_clean7.csv")
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")

# ignore standing water type and stream type categories because they do not require time series
df = df[~df["Category"].isin(["standing water type", "stream type"])]

# months per spot
spot_month = (
    df
    .groupby(["longitude", "latitude", "Category", "year_month"])
    .size()
    .reset_index(name="n_obs")
)

def get_all_streaks(months):
    months = sorted(set(months))
    streaks = []
    start = months[0]
    count = 1

    for i in range(1, len(months)):
        if months[i] == months[i-1] + 1:
            count += 1
        else:
            if count >= 12:
                streaks.append({
                    "streak_start": start,
                    "streak_end": months[i-1],
                    "streak_months": count
                })
            start = months[i]
            count = 1

    # last streak
    if count >= 12:
        streaks.append({
            "streak_start": start,
            "streak_end": months[-1],
            "streak_months": count
        })

    return pd.DataFrame(streaks)

streak_periods = (
    spot_month
    .groupby(["longitude", "latitude", "Category"])["year_month"]
    .apply(get_all_streaks)
    .reset_index(level=3, drop=True)
    .reset_index()
)

# only keep months with 12+ consecutively active months
persistent = streak_periods[streak_periods["streak_months"] >= 12].copy().reset_index(drop=True)

# country
spot_country = (
    df
    .groupby(["longitude", "latitude"])["Country"]
    .first()
    .reset_index()
)
persistent = persistent.merge(spot_country, on=["longitude", "latitude"], how="left")

# users active during the streak at this spot
df_merged = df.merge(
    persistent[["longitude", "latitude", "Category", "streak_start", "streak_end"]],
    on=["longitude", "latitude", "Category"],
    how="inner"
)

df_merged = df_merged[
    (df_merged["year_month"] >= df_merged["streak_start"]) &
    (df_merged["year_month"] <= df_merged["streak_end"])
]

spot_users = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])["created_by"]
    .nunique()
    .reset_index(name="n_users")
)
persistent = persistent.merge(spot_users, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

spot_user_ids = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])["created_by"]
    .apply(lambda x: [int(i) for i in x.unique()])
    .reset_index(name="user_ids")
)
persistent = persistent.merge(spot_user_ids, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

spot_obs = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])
    .size()
    .reset_index(name="n_obs")
)
persistent = persistent.merge(spot_obs, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

# final table
persistent = persistent[["latitude", "longitude", "Country", "streak_months", "streak_start", "streak_end", "n_users", "user_ids", "n_obs", "Category"]]
persistent = persistent.sort_values("streak_months", ascending=False).reset_index(drop=True)

print(persistent.head(20))
print(f"number of spots with >=12 months consecutive activity: {len(persistent)}")

persistent.to_csv("../Products/CSVs/persistent_spots_monthly.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_52116\991419436.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4

     latitude   longitude         Country  streak_months streak_start  \
0   47.789612   13.068625         Austria           74.0      2018-08   
1   51.580956   -0.873880  United Kingdom           73.0      2020-04   
2   51.608782   -0.881946  United Kingdom           73.0      2020-04   
3   51.574168   -0.871882  United Kingdom           73.0      2020-04   
4   51.602152   -0.880616  United Kingdom           73.0      2020-04   
5   51.602487   -0.880738  United Kingdom           73.0      2020-04   
6   51.562064   -0.867595  United Kingdom           73.0      2020-04   
7   51.581133   -0.873914  United Kingdom           73.0      2020-04   
8   47.394939    8.733537     Switzerland           72.0      2020-05   
9   51.571622   -0.870799  United Kingdom           69.0      2020-04   
10  47.395523    8.730670     Switzerland           66.0      2020-11   
11  52.193365    9.082108         Germany           63.0      2020-09   
12  51.060439 -115.328485          Canada          

Now, for both of these, show user, the spots they managed, and the weighted number of spots

In [14]:
def persistent_user(persistent, category):
    # load user_ids as lists
    persistent["user_ids"] = persistent["user_ids"].apply(ast.literal_eval)

    # collect spots per user
    user_spots = {}

    for _, row in persistent.iterrows():
        spot = (row["latitude"], row["longitude"], row["Category"], row["streak_start"], row["streak_end"])
        n_users = row["n_users"]
        for user in row["user_ids"]:
            if user not in user_spots:
                user_spots[user] = {"spots": [], "weighted_contribution": 0}
            user_spots[user]["spots"].append(spot)
            user_spots[user]["weighted_contribution"] += 1 / n_users  # percentage

    # create dataframe
    user_table = pd.DataFrame([
        {
            "user_id": user,
            "n_spots": len(data["spots"]),
            "spots": data["spots"],
            "weighted_spots": round(data["weighted_contribution"], 3)
        }
        for user, data in user_spots.items()
    ]).sort_values("weighted_spots", ascending=False).reset_index(drop=True)

    if category == "14d":
        user_table.to_csv("../Products/CSVs/persistent_users_14d.csv", index=False)
    elif category == "monthly":
        user_table.to_csv("../Products/CSVs/persistent_users_monthly.csv", index=False)
    else:
        raise ValueError
    return user_table

In [15]:
persistent_14d = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
persistent_monthly = pd.read_csv("../Products/CSVs/persistent_spots_monthly.csv")

persistent_user_14d = persistent_user(persistent_14d, "14d")
persistent_user_monthly = persistent_user(persistent_monthly, "monthly")

Create a timeline of the number of spots starting, ending and currently active, per year for both location sets

In [23]:
persistent_14d = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
persistent_monthly = pd.read_csv("../Products/CSVs/persistent_spots_monthly.csv")
for df in [persistent_14d, persistent_monthly]:
    df["streak_start"] = pd.to_datetime(df["streak_start"])
    df["streak_end"] = pd.to_datetime(df["streak_end"])

def calc_annual(persistent):
    results = []
    for year in range(2017, 2027):
        year_start = pd.Timestamp(f"{year}-01-01")
        year_end = pd.Timestamp(f"{year}-12-31")
        if year != 2026:
            active = persistent[(persistent["streak_start"] <= year_end) & (persistent["streak_end"] > year_end)]
        else:
            active = persistent[(persistent["streak_start"] <= year_end) & (persistent["streak_end"] >= pd.Timestamp("2026-04-01"))]
        new = persistent[(persistent["streak_start"] >= year_start) & (persistent["streak_start"] <= year_end)]
        ending = persistent[
            (persistent["streak_end"] >= year_start) &
            (persistent["streak_end"] <= year_end) &
            (persistent["streak_end"] < pd.Timestamp("2026-04-01"))  # still active spots
        ]
        results.append({"year": year, "active": len(active), "new": len(new), "ending": len(ending)})
    return pd.DataFrame(results)

df_14d = calc_annual(persistent_14d)
df_monthly = calc_annual(persistent_monthly)


fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

for ax, df, title in zip(axes, [df_14d, df_monthly],
                          ["a) Bi-weekly Persistent Spots (≤14 days gap, ≥180 days)",
                           "b) Monthly Persistent Spots (≤1 month gap, ≥12 months)"]):
    ax.plot(df["year"], df["active"], color="teal", linewidth=2, marker="o", label="Active Streaks")
    ax.bar(df["year"] - 0.2, df["new"], width=0.35, color="lightblue", label="New Streaks", alpha=0.8)
    ax.bar(df["year"] + 0.2, df["ending"], width=0.35, color="salmon", label="Ending Streaks", alpha=0.8)
    ax.set_ylabel("Number of Spots", fontsize=18)
    ax.set_title(title, fontsize=21, fontweight="bold", loc="left")
    ax.legend(fontsize=14)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', labelsize=14)

axes[1].set_xlabel("Year", fontsize=18)
x_labels = ["2017 (from Feb)","2018","2019","2020","2021","2022","2023","2024","2025","2026 (to Apr)"]
axes[1].set_xticks(range(2017,2027), x_labels, fontsize=14)

plt.tight_layout()
plt.savefig("../Products/persistent_spots_annual_combined.png", dpi=300, bbox_inches="tight")
plt.close()

In [19]:
df_monthly

,year,active,new,ending
0,2017,4,4,0
1,2018,10,7,1
2,2019,25,19,4
3,2020,54,42,13
4,2021,93,44,5
5,2022,60,6,39
6,2023,61,13,12
7,2024,36,5,30
8,2025,11,1,26
9,2026,11,0,0


Create a maps of these two persistent spot criteria.

In [24]:
def persistent_mapper(filename):
    persistent = pd.read_csv(f"../Products/CSVs/{filename}.csv")
    world = gpd.read_file("../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")

    category_colors = {
        "physical scale": "#1418fc",
        "plastic pollution": "#fbff2b",
        "soil moisture": "#82571b",
        "standing water type": "#dd6ef0",
        "stream type": "#fc1471",
        "temporary stream": "#8c14fc",
        "virtual scale": "#14c2fc"
    }

    gdf = gpd.GeoDataFrame(
        persistent,
        geometry=gpd.points_from_xy(persistent["longitude"], persistent["latitude"]),
        crs="EPSG:4326"
    ).to_crs("+proj=robin")
    gdf = gdf.sort_values("n_users", ascending=False)

    fig, ax = plt.subplots(1, 1, figsize=(16, 8))

    world.to_crs("+proj=robin").plot(
        ax=ax,
        color="grey",
        edgecolor="black",
        linewidth=0.5
    )

    for cat, color in category_colors.items():
        subset = gdf[gdf["Category"] == cat]
        if len(subset) == 0:
            continue
        ax.scatter(
            subset.geometry.x,
            subset.geometry.y,
            s=np.log1p(subset["n_users"]) * 70,
            color=color,
            edgecolor="black",
            linewidth=0.3,
            alpha=0.8,
            label=cat,
            zorder=2
        )

    # legend categories
    legend_cat_handles = [
        plt.scatter([], [], s=50, color=color, edgecolor="black", linewidth=0.3, alpha=0.8, label=cat.title())
        for cat, color in category_colors.items()
        if cat in gdf["Category"].unique()
    ]
    legend_cats = ax.legend(
        handles=legend_cat_handles,
        title="Category",
        loc="lower left",
        bbox_to_anchor=(0, 0.22),
        frameon=True,
        title_fontsize=15,
        fontsize=12
    )

    # legend user counts
    user_counts = [1, 5, 10, 20]
    legend_handles = [
        plt.scatter([], [], s=np.log1p(u) * 70, color="lightgrey",
                    edgecolor="black", linewidth=0.3, alpha=0.8, label=str(u))
        for u in user_counts
    ]
    legend_users = ax.legend(
        handles=legend_handles,
        title="Number of Unique Users",
        title_fontsize=15,
        fontsize=12,
        loc="lower left",
        frameon=True,
    )
    ax.add_artist(legend_cats)  # show both legends

    if "14d" in filename:
        ax.set_title("Persistent Spots (Every 14 Days for ≥180 Days)", fontsize=18)
        ax.set_axis_off()
        plt.savefig("../Products/persistent_spots_map_14d.png", dpi=300, bbox_inches="tight")
    else:
        ax.set_title("Persistent Spots (Every Month for ≥12 Consecutive Months)", fontsize=18)
        ax.set_axis_off()
        plt.savefig("../Products/persistent_spots_map_monthly.png", dpi=300, bbox_inches="tight")
    plt.close()

In [25]:
for file in ["persistent_spots_monthly", "persistent_spots_14d"]:
    persistent_mapper(file)

A few stats on the persistent spots

In [36]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_monthly.csv")

df1_countries = df1.groupby("Country").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df2_countries = df2.groupby("Country").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df_combined = df1_countries.merge(df2_countries, on="Country", how="outer", suffixes=("_14d", "_monthly")).fillna(0).astype({"n_spots_14d": int, "n_spots_monthly": int}).sort_values("Country", ascending=True)

df_combined

,Country,n_spots_14d,n_spots_monthly
0,Australia,1,0
1,Austria,13,12
2,Canada,1,1
3,Chile,2,2
4,France,0,20
5,Germany,19,10
6,Ireland,0,2
7,Kyrgyzstan,1,3
8,Portugal,0,1
9,Spain,1,8


In [41]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_monthly.csv")

df1_countries = df1.groupby("Category").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df2_countries = df2.groupby("Category").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df_combined = df1_countries.merge(df2_countries, on="Category", how="outer", suffixes=("_14d", "_monthly")).fillna(0).astype({"n_spots_14d": int, "n_spots_monthly": int}).sort_values("n_spots_14d", ascending=False)

df_combined

,Category,n_spots_14d,n_spots_monthly
2,temporary stream,43,93
3,virtual scale,37,35
1,soil moisture,8,5
0,physical scale,7,8


In [44]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_monthly.csv")
merged = df1.merge(df2, on=["latitude", "longitude"], how="inner")
in_both = df1.merge(df2, on=["latitude", "longitude"], how="inner")

print(f"Number of identical spots: {len(merged)}")
print(f"Set 1 total: {len(df1)}")
print(f"Set 2 total: {len(df2)}")
print(f"Set 1 spots that are also in set 2: {len(in_both)} ({len(in_both)/len(df1)*100:.1f}%)")
print(f"Set 2 spots that are also in set 1: {len(in_both)} ({len(in_both)/len(df2)*100:.1f}%)")
merged

Number of identical spots: 85
Set 1 total: 95
Set 2 total: 141
Set 1 spots that are also in set 2: 85 (89.5%)
Set 2 spots that are also in set 1: 85 (60.3%)


,latitude,longitude,Country_x,streak_days,streak_start_x,streak_end_x,n_users_x,user_ids_x,n_obs_x,Category_x,Country_y,max_streak,streak_start_y,streak_end_y,n_users_y,user_ids_y,n_obs_y,Category_y
0,49.375012,8.888853,Germany,1822.0,2020-07-28,2025-07-24,1,[30178],1785,physical scale,Germany,61.0,2020-07,2025-07,1,[30178],1785,physical scale
1,49.375062,8.889207,Germany,1822.0,2020-07-28,2025-07-24,1,[30178],1761,physical scale,Germany,61.0,2020-07,2025-07,1,[30178],1761,physical scale
2,51.060439,-115.328485,Canada,1763.0,2019-08-02,2024-05-30,2,"[15185, 93483]",1654,temporary stream,Canada,63.0,2019-04,2024-06,2,"[15185, 93483]",1743,temporary stream
3,47.726072,13.065014,Austria,1410.0,2017-12-25,2021-11-04,3,"[2316, 4264, 5659]",1123,virtual scale,Austria,54.0,2017-12,2022-05,3,"[2316, 4264, 5659]",1155,virtual scale
4,47.394939,8.733537,Switzerland,791.0,2022-07-11,2024-09-09,1,[28919],749,virtual scale,Switzerland,72.0,2020-05,2026-04,1,[28919],1927,virtual scale
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,44.235586,-69.100391,United States of America,203.0,2023-11-13,2024-06-03,1,[82037],45,virtual scale,United States of America,15.0,2023-06,2024-08,1,[82037],95,virtual scale
81,46.461532,6.837685,Switzerland,199.0,2022-01-01,2022-07-19,1,[39422],82,virtual scale,Switzerland,43.0,2021-05,2024-11,2,"[39422, 52314]",379,virtual scale
82,47.985406,7.959071,Germany,198.0,2019-01-14,2019-07-31,1,[11290],45,virtual scale,Germany,18.0,2019-01,2020-06,1,[11290],187,virtual scale
83,46.973372,7.477991,Switzerland,196.0,2021-01-23,2021-08-07,1,[3065],40,physical scale,Switzerland,15.0,2020-12,2022-02,1,[3065],77,physical scale
